# VAE vs GAN: Comparative Experiment

This notebook provides a structured framework to compare Variational Autoencoders (VAE) and Generative Adversarial Networks (GAN) trained on cat images. We focus on four key areas:
1. **Visual Quality**: Sharpness vs. Blurriness.
2. **FID Score**: Objective image quality metrics.
3. **Latent Space Smoothness**: How well the models interpolate between points.
4. **Reconstruction**: VAE's ability to encode and decode real images.

## 1. Imports and Model Loading

In [1]:
import os
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

sys.path.append(os.path.abspath('..'))

from scripts import (
    build_vae,
    build_gan,
    compute_fid,
    vae_sample_iterator,
    gan_sample_iterator,
    interpolate_vae,
    interpolate_gan,
    sample_vae_latents,
    sample_gan_latents,
    prepare_data,
    DataFixedConfig,
    VAEFixedConfig,
    GANFixedConfig,
    denormalize
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_checkpoint(path, model_type):
    ckpt = torch.load(path, map_location=DEVICE)
    params = ckpt['model_params']
    fixed = ckpt.get('fixed_params', None)
    
    # Note: data fixed config might need to be consistent with training
    data_fixed = DataFixedConfig(image_size=64, channels=3)
    
    if model_type == 'vae':
        model_fixed = VAEFixedConfig()
        model = build_vae(params, model_fixed, data_fixed)
        model.load_state_dict(ckpt['model_state_dict'])
    else:
        model_fixed = GANFixedConfig(spectral_norm=ckpt.get('spectral_norm', True))
        # build_gan returns (gen, disc)
        model, _ = build_gan(params, model_fixed, data_fixed)
        model.load_state_dict(ckpt['generator_state_dict'])
        
    model.to(DEVICE)
    model.eval()
    return model, ckpt

# Update these paths to your best runs
VAE_PATH = "../reports/runs/vae/20260613_233012_vae_z256_b0_8_lr0_0005_e80/checkpoints/vae_final.pt"
GAN_PATH = "../reports/runs/gan/20260615_003303_gan_z128_g64_d64_lr0_0002_ls0_1_e200/checkpoints/gan_final.pt"

vae, vae_ckpt = load_checkpoint(VAE_PATH, 'vae')
gan, gan_ckpt = load_checkpoint(GAN_PATH, 'gan')

print("Models loaded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '../reports/runs/vae/20260613_233012_vae_z256_b0_8_lr0_0005_e80/checkpoints/vae_final.pt'

## 2. Visual Comparison: Blurriness vs. Sharpness

We generate a grid of samples from both models to visually inspect the trade-off. VAEs are expected to be smoother but blurrier, while GANs should be sharper but might have more structural artifacts.

In [ ]:
def show_samples(model, model_type, num=16):
    with torch.no_grad():
        if model_type == 'vae':
            z = torch.randn(num, model.latent_dim).to(DEVICE)
            samples = model.decode(z)
        else:
            z = torch.randn(num, model.latent_dim, 1, 1).to(DEVICE)
            samples = model(z)
            
    samples = denormalize(samples).cpu().permute(0, 2, 3, 1).numpy()
    
    fig, axes = plt.subplots(num//4, 4, figsize=(10, num//4 * 2.5))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(samples[i])
        ax.axis('off')
    plt.suptitle(f"{model_type.upper()} Samples")
    plt.show()

show_samples(vae, 'vae')
show_samples(gan, 'gan')

## 3. Objective Quality: FID Score

FID (Fréchet Inception Distance) measures how similar the distribution of generated images is to the real ones. Lower is better.

In [ ]:
DATA_FIXED = DataFixedConfig(data_dir="../data/cats", image_size=64, channels=3)
NUM_FID_SAMPLES = 1000
BATCH_SIZE = 64

vae_iter = vae_sample_iterator(vae, NUM_FID_SAMPLES, BATCH_SIZE, DEVICE, seed=42)
vae_fid = compute_fid(vae_iter, DATA_FIXED, num_real_samples=NUM_FID_SAMPLES, seed=42, device=DEVICE)
print(f"VAE FID: {vae_fid:.2f}")

gan_iter = gan_sample_iterator(gan, NUM_FID_SAMPLES, BATCH_SIZE, DEVICE, seed=42)
gan_fid = compute_fid(gan_iter, DATA_FIXED, num_real_samples=NUM_FID_SAMPLES, seed=42, device=DEVICE)
print(f"GAN FID: {gan_fid:.2f}")

## 4. Latent Space Smoothness: Interpolation

We pick two points in the latent space and generate images along the linear path between them. This shows if the model has learned a meaningful, continuous representation.

In [ ]:
def plot_interpolation(images, title):
    images = denormalize(images).cpu().permute(0, 2, 3, 1).numpy()
    fig, axes = plt.subplots(1, len(images), figsize=(15, 2))
    for i, ax in enumerate(axes):
        ax.imshow(images[i])
        ax.axis('off')
    plt.suptitle(title)
    plt.show()

z1_vae, z2_vae = sample_vae_latents(vae, DEVICE, seed=123)
interp_vae = interpolate_vae(vae, z1_vae, z2_vae, num_steps=10)
plot_interpolation(interp_vae, "VAE Latent Interpolation")

z1_gan, z2_gan = sample_gan_latents(gan, DEVICE, seed=123)
interp_gan = interpolate_gan(gan, z1_gan, z2_gan, num_steps=10)
plot_interpolation(interp_gan, "GAN Latent Interpolation")

## 5. VAE Reconstruction: Encoding Reality

Unlike the standard DCGAN, the VAE has an encoder. We can take a real image, encode it to a latent vector, and see how well the model can reconstruct it.

In [ ]:
from scripts import prepare_data, build_dataloaders

prepared = prepare_data(DATA_FIXED, train_fraction=0.9, validation_fraction=0.1, augment_flip=False, seed=42)
_, val_loader = build_dataloaders(prepared, DATA_FIXED, batch_size=8, seed=42)

real_batch = next(iter(val_loader)).to(DEVICE)
with torch.no_grad():
    recon, mu, logvar = vae(real_batch)

real_imgs = denormalize(real_batch).cpu().permute(0, 2, 3, 1).numpy()
recon_imgs = denormalize(recon).cpu().permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(15, 4))
for i in range(8):
    axes[0, i].imshow(real_imgs[i])
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_imgs[i])
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", size='large')
axes[1, 0].set_ylabel("Reconstructed", size='large')
plt.suptitle("VAE Reconstruction of Real Images")
plt.show()

## Summary of Observations

*Write your conclusions here based on the results above.*